# CTRNN Training & PID Analysis — Both Tasks
Train a CTRNN on **PerceptualDecisionMaking** (low integration demand) and **ContextDecisionMaking** (high integration demand), run time-resolved Gaussian PID analysis on each, and produce comparison plots.

Mirrors the full training pipeline notebook structure:  
`load NPZ → train with early stopping + LR scheduler → extract hidden states → PID → plots`

**Key settings** are at the top of each section so they are easy to change without reading the whole notebook.

### 0. Imports

In [28]:
from pathlib import Path
import copy, json, os, sys, time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# ── Locate repository root and make src/ importable ──────────────────────────
repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists():
        repo_root = candidate
        break
if repo_root is not None and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.models.ctrnn       import CTRNN
from src.analysis.gaussian_pid import gaussian_pid_rnn
from src.tasks.data_loader  import load_mante_data

print(f"Repository root : {repo_root}")
print(f"PyTorch version : {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute device  : {device.type.upper()}")
if device.type == "cuda":
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

Repository root : e:\TUM\3sem\NeuroAI\project\github_neuroai
PyTorch version : 2.5.1+cu121
Compute device  : CUDA
GPU             : NVIDIA GeForce GTX 1650 with Max-Q Design


### 1. Parameters
All top-level settings live here. Change `HIDDEN_SIZE`, `BASE_PATH`, or `N_BIPARTITIONS` and re-run — nothing else needs editing.

In [29]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_PATH   = Path(r"E:\TUM\3sem\NeuroAI\project\data_generation\seed42")
RESULTS_DIR = Path(r"E:\TUM\3sem\NeuroAI\project\github_neuroai\results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR = RESULTS_DIR / "weights"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Model ─────────────────────────────────────────────────────────────────────
HIDDEN_SIZE    = 100      # hidden units in the CTRNN
OUTPUT_DIM     = 3        # fixate / choice1 / choice2
DT             = 20       # ms per timestep (used for PID eval timing labels)

# ── Data ──────────────────────────────────────────────────────────────────────
BATCH_SIZE      = 1024 * 2
SUBSAMPLE_STEP  = 1      # raw trials dt=1ms → effective 10ms (750→75 steps)
TEST_SPLIT_FMT  = "test_uniform"   # test NPZ filename (no .npz)

# ── Training ──────────────────────────────────────────────────────────────────
NUM_EPOCHS = 50
PATIENCE   = 10           # early-stopping patience (epochs)
LR         = 1e-3

# LR scheduler (ReduceLROnPlateau) — task-specific plateau patience
LR_SCHED = {
    "perceptual": {"lr_factor": 0.5, "lr_patience": 3, "lr_min": 1e-5},
    "context":    {"lr_factor": 0.5, "lr_patience": 6, "lr_min": 1e-5},
}

# ── PID ───────────────────────────────────────────────────────────────────────
N_BIPARTITIONS = 200      # random 50/50 splits averaged per timestep
DEFAULT_SEED   = 42

# ── Task metadata ─────────────────────────────────────────────────────────────
TASK_META = {
    "perceptual": {"input_dim": 3, "npz_task": "perceptual"},
    "context":    {"input_dim": 7, "npz_task": "context"},
}

PALETTE = {
    "synergy":    "#54A24B",
    "redundancy": "#4C78A8",
    "unique1":    "#F58518",
    "unique2":    "#E45756",
    "mi_joint":   "#020202",
    "stim_end":   "#CC0000",
    "train":      "#2196F3",
    "val":        "#FF9800",
}

print("Parameters set.")

Parameters set.


### 2. Architecture
Thin wrapper around `src/models/ctrnn.py` so the forward pass always returns `(outputs, hidden_states)` in `(Batch, Time, Features)` layout. Hidden states are what PID operates on — not the output logits.

In [30]:
class WrappedCTRNN(nn.Module):
    def __init__(self, input_dim, hidden_size, output_size=OUTPUT_DIM):
        super().__init__()
        self.model = CTRNN(input_size=input_dim,
                           hidden_size=hidden_size,
                           output_size=output_size)

    def forward(self, x):
        """x: (B, T, F) → outputs (B,T,C), hidden_states (B,T,H)"""
        outputs, _, hidden_states = self.model(x, return_dynamics=True)
        return outputs, hidden_states

### 3. Loss and accuracy helpers
Loss is computed **only on decision-period timesteps** (`periods == 2`), matching the notebook convention. Accuracy counts correct predictions over those same timesteps.

In [31]:
def masked_cross_entropy(outputs, targets, mask):
    """Cross-entropy over decision-period timesteps only."""
    if mask.sum() == 0:
        return torch.tensor(0.0, requires_grad=True, device=outputs.device)
    return F.cross_entropy(outputs[mask], targets[mask])


def masked_accuracy_counts(outputs, targets, mask):
    """Return (n_correct, n_total) over decision-period timesteps."""
    if mask.sum() == 0:
        return 0, 0
    preds   = outputs[mask].argmax(dim=1)
    correct = (preds == targets[mask]).sum().item()
    return correct, int(mask.sum().item())

### 4. Training loop
Identical structure to the full training pipeline notebook:
- Decision-period masked cross-entropy loss
- `ReduceLROnPlateau` scheduler (task-specific patience)
- Early stopping on validation loss (patience = 10)
- Best-checkpoint weights restored at the end
- Train **and** validation accuracy tracked every epoch

In [32]:
def train_ctrnn(model, train_loader, val_loader, device,
                num_epochs=NUM_EPOCHS, lr=LR, patience=PATIENCE,
                lr_factor=0.5, lr_patience=5, lr_min=1e-5):
    """
    Train a single CTRNN and return (history, best_state_dict).
    history keys: train_loss, train_acc, val_loss, val_acc, lr
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=lr_factor,
        patience=lr_patience, min_lr=lr_min,
    )

    best_val_loss    = float("inf")
    best_state       = copy.deepcopy(model.state_dict())
    patience_counter = 0
    history = {"train_loss": [], "train_acc": [],
               "val_loss":   [], "val_acc":   [], "lr": []}
    train_start = time.time()

    for epoch in range(num_epochs):
        epoch_start = time.time()

        # ── Training phase ──────────────────────────────────────────────────
        model.train()
        run_loss, n_batches       = 0.0, 0
        train_correct, train_total = 0, 0

        for obs, labels, periods, cohs, ctxs in train_loader:
            obs, labels, periods = obs.to(device), labels.to(device), periods.to(device)
            optimizer.zero_grad()
            outputs, _ = model(obs)
            mask = (periods == 2)
            if mask.sum() > 0:
                loss = masked_cross_entropy(outputs, labels, mask)
                loss.backward()
                optimizer.step()
                run_loss += loss.item(); n_batches += 1
                c, t = masked_accuracy_counts(outputs, labels, mask)
                train_correct += c; train_total += t

        avg_train_loss = run_loss / max(1, n_batches)
        avg_train_acc  = train_correct / max(1, train_total)
        history["train_loss"].append(avg_train_loss)
        history["train_acc"].append(avg_train_acc)

        # ── Validation phase ────────────────────────────────────────────────
        model.eval()
        run_val, n_val              = 0.0, 0
        val_correct, val_total      = 0, 0
        with torch.no_grad():
            for obs, labels, periods, cohs, ctxs in val_loader:
                obs, labels, periods = obs.to(device), labels.to(device), periods.to(device)
                outputs, _ = model(obs)
                mask = (periods == 2)
                if mask.sum() > 0:
                    run_val += masked_cross_entropy(outputs, labels, mask).item()
                    n_val   += 1
                    c, t = masked_accuracy_counts(outputs, labels, mask)
                    val_correct += c; val_total += t

        avg_val_loss = run_val / max(1, n_val)
        avg_val_acc  = val_correct / max(1, val_total)
        history["val_loss"].append(avg_val_loss)
        history["val_acc"].append(avg_val_acc)

        # ── LR schedule + early stopping ────────────────────────────────────
        scheduler.step(avg_val_loss)
        history["lr"].append(optimizer.param_groups[0]["lr"])

        if avg_val_loss < best_val_loss:
            best_val_loss    = avg_val_loss
            best_state       = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch [{epoch+1:03d}/{num_epochs}] "
                  f"{time.time()-epoch_start:.1f}s | "
                  f"train {avg_train_loss:.4f} / {avg_train_acc:.3f} | "
                  f"val {avg_val_loss:.4f} / {avg_val_acc:.3f} | "
                  f"LR {optimizer.param_groups[0]['lr']:.1e}")

        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch+1} "
                  f"(best val loss={best_val_loss:.4f})")
            break

    print(f"  Training done ({time.time()-train_start:.1f}s). "
          f"Restoring best weights...")
    model.load_state_dict(best_state)
    return history, best_state


def evaluate_accuracy(model, loader, device):
    """Decision-period accuracy on any loader."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for obs, labels, periods, cohs, ctxs in loader:
            obs, labels, periods = obs.to(device), labels.to(device), periods.to(device)
            outputs, _ = model(obs)
            mask = (periods == 2)
            c, t = masked_accuracy_counts(outputs, labels, mask)
            correct += c; total += t
    return correct / max(1, total)

### 5. Learning curve helper

In [33]:
def plot_learning_curve(history, title, save_path=None):
    """Two-panel: loss (top) and accuracy (bottom) vs epoch."""
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax_l, ax_a) = plt.subplots(2, 1, figsize=(8, 7))

    ax_l.plot(epochs, history["train_loss"], color=PALETTE["train"], lw=2, label="Train loss")
    ax_l.plot(epochs, history["val_loss"],   color=PALETTE["val"],   lw=2, ls="--", label="Val loss")
    ax_l.set_ylabel("Loss", fontsize=13); ax_l.legend(fontsize=11); ax_l.grid(alpha=0.3)

    ax_a.plot(epochs, history["train_acc"], color=PALETTE["train"], lw=2, label="Train acc")
    ax_a.plot(epochs, history["val_acc"],   color=PALETTE["val"],   lw=2, ls="--", label="Val acc")
    ax_a.set_xlabel("Epoch", fontsize=13)
    ax_a.set_ylabel("Decision accuracy", fontsize=13)
    ax_a.set_ylim(0, 1); ax_a.legend(fontsize=11); ax_a.grid(alpha=0.3)

    for ax in (ax_l, ax_a):
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(labelsize=11)

    fig.suptitle(title, fontsize=14, fontweight="bold")
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved → {save_path}")
    plt.show(); plt.close(fig)

### 6. Hidden state extraction
Mirrors `extract_hidden_trajectories` from the pipeline notebook. Runs the test loader through the model and collects `(H, Y, C, P)`:
- `H` — hidden states `(N, T, hidden_size)` — the PID input
- `Y` — signed coherence per trial `(N,)` — PID target 1
- `C` — context label per trial `(N,)` — PID target 2 (CDM only)
- `P` — period labels per trial `(N, T)` — used to find stimulus-end timestep

In [34]:
def extract_hidden_states(model, loader, device):
    model.eval()
    H_l, Y_l, C_l, P_l = [], [], [], []
    with torch.no_grad():
        for obs, labels, periods, cohs, ctxs in loader:
            obs = obs.to(device)
            _, hidden = model(obs)          # (B, T, H)
            H_l.append(hidden.cpu().numpy())
            cohs_np = cohs.numpy()
            Y_l.append(cohs_np[:, 0] if cohs_np.ndim > 1 else cohs_np)
            C_l.append(ctxs.numpy())
            P_l.append(periods.numpy())

    H = np.concatenate(H_l, axis=0)   # (N, T, H)
    Y = np.concatenate(Y_l, axis=0).astype(float)
    C = np.concatenate(C_l, axis=0)
    P = np.concatenate(P_l, axis=0)
    return H, Y, C, P


def find_stim_end(P):
    """Modal last stimulus timestep across trials (periods==1)."""
    ends = []
    for p in P:
        s = np.where(p == 1)[0]
        ends.append(s[-1] if len(s) > 0 else 0)
    return int(np.median(ends))

### 7. PID analysis
Runs `gaussian_pid_rnn` at every timestep for both PID targets of each task:
- PDM: signed coherence + binary decision label
- CDM: signed coherence of relevant modality + context label (0 or 1)

In [35]:
def run_pid_analysis(H, Y, C, P, task_key, n_bipartitions=N_BIPARTITIONS):
    stim_end = find_stim_end(P)
    results  = {}

    if task_key == "perceptual":
        targets = [("coherence", Y), ("decision", (Y > 0).astype(float))]
    else:
        targets = [("coherence", Y), ("context", C.astype(float))]

    for name, vals in targets:
        print(f"  PID | target={name} | H={H.shape[2]} | T={H.shape[1]} | N={H.shape[0]}")
        pid = gaussian_pid_rnn(
            activations=H, target=vals, timestep=None,
            bipartitions="random", n_bipartitions=n_bipartitions,
            seed=DEFAULT_SEED, log_base=2, regularization=1e-5,
        )
        results[name] = {
            "synergy":    np.array(pid["synergy"]),
            "redundancy": np.array(pid["redundancy"]),
            "unique1":    np.array(pid["unique1"]),
            "unique2":    np.array(pid["unique2"]),
            "mi_joint":   np.array(pid["mi_joint"]),
            "stim_end": stim_end, "seq_len": H.shape[1],
        }
        tend = stim_end
        print(f"    stim_end=t{tend} | "
              f"Syn={results[name]['synergy'][tend]:.4f}b | "
              f"Red={results[name]['redundancy'][tend]:.4f}b")
    return results


def _pid_ax(ax, pid, title, stim_end=None):
    T    = pid["seq_len"]
    t    = np.arange(T)
    tend = stim_end if stim_end is not None else pid["stim_end"]
    ax.plot(t, pid["mi_joint"],   label="Total MI",   color=PALETTE["mi_joint"],   lw=2,   ls=":")
    ax.plot(t, pid["synergy"],    label="Synergy",    color=PALETTE["synergy"],    lw=3)
    ax.plot(t, pid["redundancy"], label="Redundancy", color=PALETTE["redundancy"], lw=3)
    ax.plot(t, pid["unique1"],    label="Unique 1",   color=PALETTE["unique1"],    lw=1.5, alpha=0.8)
    ax.plot(t, pid["unique2"],    label="Unique 2",   color=PALETTE["unique2"],    lw=1.5, alpha=0.8)
    ax.axvline(tend, color=PALETTE["stim_end"], ls="--", lw=2, label=f"Stim end (t={tend})")
    ax.axvspan(0, tend, color="gray", alpha=0.08)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel(f"Timestep (dt={DT}ms)"); ax.set_ylabel("Information (bits)")
    ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.3)

### 8. Load data and train
Loads train / val / test NPZ files for both tasks and trains one CTRNN per task. Each model is saved to `WEIGHTS_DIR`.

In [36]:
TASKS = ["perceptual", "context"]

trained   = {}   # task → model
histories = {}   # task → history dict

for task in TASKS:
    meta     = TASK_META[task]
    npz_base = BASE_PATH / task

    print(f"\n{'='*65}")
    print(f"  TASK: {task.upper()}  (input_dim={meta['input_dim']})")
    print(f"{'='*65}")

    train_loader = load_mante_data(
        str(npz_base / "train.npz"),
        batch_size=BATCH_SIZE, shuffle=True, subsample_step=SUBSAMPLE_STEP,
        input_dim=meta["input_dim"],
    )
    val_loader = load_mante_data(
        str(npz_base / "val.npz"),
        batch_size=BATCH_SIZE, shuffle=False, subsample_step=SUBSAMPLE_STEP,
        input_dim=meta["input_dim"],
    )

    model = WrappedCTRNN(meta["input_dim"], HIDDEN_SIZE).to(device)
    history, best_state = train_ctrnn(
        model, train_loader, val_loader, device,
        num_epochs=NUM_EPOCHS, patience=PATIENCE, lr=LR,
        **LR_SCHED[task],
    )

    # Save weights
    weights_path = WEIGHTS_DIR / f"ctrnn_{task}_h{HIDDEN_SIZE}.pth"
    torch.save(best_state, weights_path)
    print(f"  Weights saved → {weights_path}")

    trained[task]   = model
    histories[task] = history

    # Learning curve
    fig_path = RESULTS_DIR / f"learning_curve_{task}.png"
    plot_learning_curve(history, title=f"CTRNN | {task.capitalize()} task",
                        save_path=fig_path)

print("\nAll models trained.")


  TASK: PERCEPTUAL  (input_dim=3)
  Epoch [001/50] 19.0s | train 0.4803 / 0.757 | val 0.2801 / 0.880 | LR 1.0e-03
  Epoch [005/50] 17.0s | train 0.2593 / 0.885 | val 0.2515 / 0.890 | LR 1.0e-03
  Epoch [010/50] 17.2s | train 0.2538 / 0.888 | val 0.2486 / 0.890 | LR 1.0e-03


KeyboardInterrupt: 

### 9. Test-set evaluation

In [ ]:
test_metrics = {}

for task in TASKS:
    meta     = TASK_META[task]
    npz_base = BASE_PATH / task
    test_loader = load_mante_data(
        str(npz_base / f"{TEST_SPLIT_FMT}.npz"),
        batch_size=BATCH_SIZE, shuffle=False, subsample_step=SUBSAMPLE_STEP,
        input_dim=meta["input_dim"],
    )
    acc = evaluate_accuracy(trained[task], test_loader, device)
    test_metrics[task] = acc
    print(f"  {task.capitalize():>12}  test accuracy = {acc:.4f}  ({acc*100:.1f}%)")

### 10. Extract hidden states from test set
The test set is passed through each model once (no weight updates). Hidden states are the input to the PID analysis.

In [ ]:
hidden_data = {}   # task → (H, Y, C, P)

for task in TASKS:
    meta     = TASK_META[task]
    npz_base = BASE_PATH / task
    test_loader = load_mante_data(
        str(npz_base / f"{TEST_SPLIT_FMT}.npz"),
        batch_size=BATCH_SIZE, shuffle=False, subsample_step=SUBSAMPLE_STEP,
        input_dim=meta["input_dim"],
    )
    model = trained[task].to("cpu")
    print(f"Extracting hidden states: {task}...")
    H, Y, C, P = extract_hidden_states(model, test_loader, "cpu")
    hidden_data[task] = (H, Y, C, P)
    print(f"  H={H.shape}  Y range=[{Y.min():.1f}, {Y.max():.1f}]")

### 11. Time-resolved Gaussian PID
Compute PID at every timestep. `N_BIPARTITIONS` random 50/50 unit splits are averaged to get stable estimates. This is the slowest step — 200 bipartitions × 75 timesteps × 2 tasks takes a few minutes.

In [ ]:
all_pid = {}

for task in TASKS:
    H, Y, C, P = hidden_data[task]
    print(f"\nRunning PID: {task} ...")
    all_pid[task] = run_pid_analysis(H, Y, C, P, task, n_bipartitions=N_BIPARTITIONS)

print("\nPID analysis complete.")

### 12. 4-panel information geometry comparison
Mirrors the final figure from the companion notebook — 2 tasks × 2 PID targets. According to the project hypothesis:
- **Synergy** should peak at the end of the stimulus period (red dashed line)
- The **context task** (bottom row) should show higher synergy than the perceptual task (top row)

In [ ]:
stim_end_per = all_pid["perceptual"]["coherence"]["stim_end"]
stim_end_ctx = all_pid["context"]["coherence"]["stim_end"]

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle(
    f"Information Geometry: Task Comparison  (CTRNN, hidden={HIDDEN_SIZE})",
    fontsize=16, fontweight="bold", y=0.98,
)

panel_map = [
    (0, 0, "perceptual", "coherence", "Perceptual | Coherence target",  stim_end_per),
    (0, 1, "perceptual", "decision",  "Perceptual | Decision target",   stim_end_per),
    (1, 0, "context",    "coherence", "Context | Coherence target",     stim_end_ctx),
    (1, 1, "context",    "context",   "Context | Context-rule target",  stim_end_ctx),
]

for row, col, task, tgt, title, tend in panel_map:
    ax = axes[row, col]
    _pid_ax(ax, all_pid[task][tgt], title, stim_end=tend)
    if col > 0:
        ax.get_legend().remove()
    if col == 0:
        ax.set_ylabel("Information (bits)", fontsize=11)
    if row == 0:
        ax.set_xlabel("")

axes[0, 0].legend(loc="upper left", fontsize=10, framealpha=0.9)

plt.tight_layout(rect=[0, 0, 1, 0.95])
save_path = RESULTS_DIR / "comparison_4panel.png"
fig.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved → {save_path}")
plt.show()

### 13. Save raw PID arrays and training metrics

In [ ]:
npz_path  = RESULTS_DIR / "results.npz"
save_dict = {}

for task in TASKS:
    for tgt, data in all_pid[task].items():
        for atom in ("synergy", "redundancy", "unique1", "unique2", "mi_joint"):
            save_dict[f"{task}_{tgt}_{atom}"] = data[atom]
        save_dict[f"{task}_{tgt}_stim_end"] = np.array(data["stim_end"])
    h = histories[task]
    for key in ("train_loss", "train_acc", "val_loss", "val_acc", "lr"):
        save_dict[f"{task}_{key}"] = np.array(h[key])
    save_dict[f"{task}_test_acc"] = np.array(test_metrics[task])

np.savez_compressed(str(npz_path), **save_dict)
print(f"Results saved → {npz_path}")
print("\nKeys saved:")
for k in sorted(save_dict.keys()):
    print(f"  {k}: {np.array(save_dict[k]).shape}")